# FLUX.2 Klein 4B — Colab

Швидка нова модель від Black Forest Labs (Apache 2.0). Підходить для **безкоштовного Colab T4** (~13GB VRAM з CPU offload).

1. **Runtime → Change runtime type → GPU (T4)**
2. **Runtime → Run all**
3. Дочекайся посилання Gradio (`Running on public URL`)

Перший запуск качає модель (~8–12 хв). Далі генерація ~5–20 с на 1024×1024.

> Це **не Fooocus**. Flux — інша архітектура. Negative prompt майже не потрібен — пиши детальний positive prompt.

In [ ]:
#@title 1) GPU check
import subprocess

r = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
if r.returncode != 0 or "GPU" not in (r.stdout or ""):
    raise RuntimeError("GPU не знайдено. Runtime → Change runtime type → T4 GPU → Restart → Run all")
print("OK:", (r.stdout or "").strip().split("\n")[0])

In [ ]:
#@title 2) Install (keep Colab torch; only add Flux stack)
import subprocess, sys

pkgs = [
    "-q",
    "transformers>=4.46.0",
    "accelerate",
    "safetensors",
    "sentencepiece",
    "protobuf",
    "huggingface_hub",
    "gradio",
    "git+https://github.com/huggingface/diffusers.git",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-U", *pkgs], check=True)

import torch
print("deps ok | torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
#@title 3) Load FLUX.2 Klein 4B
import torch
from diffusers import Flux2KleinPipeline

assert torch.cuda.is_available(), "Need GPU runtime"

major, _ = torch.cuda.get_device_capability(0)
# T4 = 7.5 -> float16; Ampere+ (8.0) -> bfloat16
dtype = torch.bfloat16 if major >= 8 else torch.float16
print(f"GPU: {torch.cuda.get_device_name(0)} | dtype={dtype}")

MODEL_ID = "black-forest-labs/FLUX.2-klein-4B"

pipe = Flux2KleinPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
)
pipe.enable_model_cpu_offload()
try:
    pipe.vae.enable_slicing()
except Exception:
    pass

print("model loaded:", MODEL_ID)

In [ ]:
#@title 4) Gradio UI
import gradio as gr
import torch

SIZES = {
    "1024x1024": (1024, 1024),
    "1152x896": (1152, 896),
    "896x1152": (896, 1152),
    "1216x832": (1216, 832),
    "832x1216": (832, 1216),
}


def generate(prompt, size_label, steps, guidance, seed):
    prompt = (prompt or "").strip()
    if not prompt:
        raise gr.Error("Enter a prompt")

    w, h = SIZES.get(size_label, (1024, 1024))
    seed = int(seed)
    if seed < 0:
        seed = int(torch.randint(0, 2**31 - 1, (1,)).item())

    gen = torch.Generator(device="cpu").manual_seed(seed)
    image = pipe(
        prompt=prompt,
        height=h,
        width=w,
        guidance_scale=float(guidance),
        num_inference_steps=int(steps),
        generator=gen,
    ).images[0]
    return image, f"seed={seed} | {w}x{h} | steps={int(steps)}"


demo = gr.Interface(
    fn=generate,
    inputs=[
        gr.Textbox(
            label="Prompt",
            lines=4,
            value=(
                "Ultra-realistic portrait photo of a young woman outdoors at golden hour, "
                "natural skin texture, sharp eyes, 85mm lens, shallow depth of field, "
                "soft cinematic light, photorealistic, no plastic skin"
            ),
        ),
        gr.Dropdown(list(SIZES.keys()), value="1024x1024", label="Size"),
        gr.Slider(1, 12, value=4, step=1, label="Steps (Klein distilled: 4 is enough)"),
        gr.Slider(0.0, 4.0, value=1.0, step=0.1, label="Guidance (usually 1.0)"),
        gr.Number(value=-1, label="Seed (-1 = random)", precision=0),
    ],
    outputs=[
        gr.Image(type="pil", label="Result"),
        gr.Textbox(label="Info"),
    ],
    title="FLUX.2 Klein 4B",
    description="New fast BFL model. Write detailed English photo-style prompts.",
    allow_flagging="never",
)

demo.queue(max_size=2).launch(share=True, debug=False)

### Tips

- **Prompt**: long and specific (camera, light, materials). Skip `masterpiece, best quality` spam.
- **Steps**: start at **4**; try 6–8 if soft.
- **Guidance**: usually **1.0**; above 2–3 can overcook.
- OOM / disconnect → smaller size or Restart session.
- Want max quality later → ComfyUI + FLUX.2 [dev] / 9B.